# Desafio 18 — Classificação de Notícias
**Grupo 18 | NLP | Classificação Multiclasse | UNIMAR 2026**

Classificação automática de manchetes de notícias em 5 categorias usando TF-IDF e três classificadores distintos.

**Dataset:** News Category Dataset (Kaggle: rmisra/news-category-dataset)

## 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import re
import json

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# NLP e Machine Learning
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Salvamento do modelo
import joblib

print("Bibliotecas importadas com sucesso!")


## 2. Carregamento e Seleção dos Dados

In [ ]:
# ---------------------------------------------------------------
# ATENÇÃO: ajuste o caminho abaixo conforme onde salvou o dataset
# Se estiver no Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# path = '/content/drive/MyDrive/News_Category_Dataset_v3.json'
#
# Se fez upload direto no Colab:
# path = '/content/News_Category_Dataset_v3.json'
# ---------------------------------------------------------------

path = 'News_Category_Dataset_v3.json'  # <-- ajuste aqui

# O dataset pode estar em formato JSON lines (um JSON por linha)
records = []
with open(path, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

df_full = pd.DataFrame(records)
print(f"Dataset completo: {df_full.shape}")
print(f"Colunas: {df_full.columns.tolist()}")
df_full.head(3)


In [ ]:
# Seleção das 5 categorias mais frequentes
TOP5 = df_full['category'].value_counts().head(5).index.tolist()
print("Top 5 categorias:", TOP5)

df = df_full[df_full['category'].isin(TOP5)].copy()
df = df[['headline', 'category']].dropna()
df = df[df['headline'].str.strip() != '']
df = df.reset_index(drop=True)

print(f"\nDataset filtrado: {df.shape}")
print("\nDistribuição das categorias:")
print(df['category'].value_counts())


## 3. Análise Exploratória (EDA)

In [ ]:
# --- Distribuição das categorias (gráfico de barras) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['category'].value_counts()

axes[0].bar(counts.index, counts.values, color=['#1F4E79','#2E75B6','#4BACC6','#70AD47','#ED7D31'])
axes[0].set_title('Distribuição das Categorias', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Categoria')
axes[0].set_ylabel('Quantidade de manchetes')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

# Tamanho médio das manchetes por categoria
df['n_palavras'] = df['headline'].apply(lambda x: len(x.split()))
media_palavras = df.groupby('category')['n_palavras'].mean().reindex(counts.index)

axes[1].barh(media_palavras.index, media_palavras.values, color='#2E75B6')
axes[1].set_title('Tamanho Médio das Manchetes (palavras)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nº médio de palavras')
for i, v in enumerate(media_palavras.values):
    axes[1].text(v + 0.05, i, f'{v:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_distribuicao.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nDesbalanceamento detectado:")
print(f"  Maior classe: {counts.max()} ({counts.idxmax()})")
print(f"  Menor classe: {counts.min()} ({counts.idxmin()})")
print(f"  Razão: {counts.max()/counts.min():.1f}x")
print("\n>> AÇÃO: class_weight='balanced' será aplicado nos modelos lineares para compensar.")


In [ ]:
# --- Palavras mais frequentes por categoria (EDA — frequência bruta) ---
# NOTA: Esta análise usa frequência bruta com stop words manuais.
# É diferente da feature importance (seção 7), que usa coeficientes do LinearSVC após TF-IDF.

from collections import Counter

STOP_WORDS_EN = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'is','was','are','were','be','been','has','have','had','will','would',
    'can','could','that','this','it','its','from','by','as','up','out',
    'into','about','after','before','not','also','than','then','when',
    'their','they','them','he','she','we','you','his','her','our','your',
    'my','i','me','us','who','what','how','new','more','over','one','two'
}

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, cat in enumerate(counts.index):
    texto = ' '.join(df[df['category'] == cat]['headline'].str.lower())
    palavras = re.findall(r'[a-z]+', texto)
    palavras_filtradas = [w for w in palavras if w not in STOP_WORDS_EN and len(w) > 2]
    top_words = Counter(palavras_filtradas).most_common(10)
    words, freqs = zip(*top_words)

    axes[idx].barh(list(words)[::-1], list(freqs)[::-1], color='#2E75B6')
    axes[idx].set_title(cat.replace(' & ', '\n& '), fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Frequência')

plt.suptitle('Top 10 Palavras por Categoria (frequência bruta, EDA)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_palavras.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Pré-processamento

In [ ]:
def limpar_texto(texto):
    """Remove pontuação e converte para lowercase."""
    texto = texto.lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['headline_limpa'] = df['headline'].apply(limpar_texto)

print("Exemplos de pré-processamento:")
for i in range(3):
    print(f"  Original : {df['headline'].iloc[i]}")
    print(f"  Limpa    : {df['headline_limpa'].iloc[i]}")
    print()


## 5. Divisão dos Dados

In [ ]:
X = df['headline_limpa']
y = df['category']

# Divisão 80% treino / 20% teste com estratificação por categoria.
# IMPORTANTE: X_test fica ISOLADO aqui e só será usado na avaliação final (seção 6).
# O StratifiedKFold (seção 6) é aplicado APENAS sobre X_train — o teste nunca vazou para o treino.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # garante proporção das classes em treino e teste
)

print(f"Treino : {X_train.shape[0]} amostras")
print(f"Teste  : {X_test.shape[0]} amostras")

print("\nDistribuição no TREINO (%):")
print((y_train.value_counts() / len(y_train) * 100).round(2))

print("\nDistribuição no TESTE (%):")
print((y_test.value_counts() / len(y_test) * 100).round(2))

print("\n>> A estratificação garante que as proporções de classes sejam similares em treino e teste.")


## 6. Treinamento e Validação Cruzada

> O `StratifiedKFold` é aplicado **apenas sobre `X_train`**. O conjunto `X_test` permanece isolado e será usado somente na avaliação final (seção 7).

In [ ]:
# Definição dos pipelines
# class_weight='balanced' compensa o desbalanceamento entre categorias (POLITICS ~35k vs TRAVEL ~9k)
pipelines = {
    'MultinomialNB': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=50000)),
        ('clf', MultinomialNB())
    ]),
    'LogisticRegression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=50000)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ]),
    'LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=50000)),
        ('clf', LinearSVC(random_state=42, max_iter=2000, class_weight='balanced'))
    ])
}

# StratifiedKFold aplicado SOMENTE sobre o conjunto de treino
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

resultados = {}
for nome, pipe in pipelines.items():
    print(f"Treinando {nome}...")
    cv_scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    resultados[nome] = {
        'Acurácia'        : cv_scores['test_accuracy'].mean(),
        'Precisão Macro'  : cv_scores['test_precision_macro'].mean(),
        'Recall Macro'    : cv_scores['test_recall_macro'].mean(),
        'F1-Score Macro'  : cv_scores['test_f1_macro'].mean(),
    }
    print(f"  F1-Score Macro: {resultados[nome]['F1-Score Macro']:.4f}")

df_resultados = pd.DataFrame(resultados).T.round(4)
print("\n=== RESULTADOS FINAIS — StratifiedKFold (k=3) — valores REAIS ===")
print(df_resultados.to_string())


In [ ]:
# Visualização dos resultados comparativos
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df_resultados.columns))
width = 0.25
cores = ['#1F4E79', '#2E75B6', '#4BACC6']

for i, (modelo, row) in enumerate(df_resultados.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=modelo, color=cores[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Métrica')
ax.set_ylabel('Valor')
ax.set_title('Comparação dos Classificadores — StratifiedKFold (k=3)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(df_resultados.columns)
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('comparacao_modelos.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Avaliação Final no Conjunto de Teste

In [ ]:
# Treinar cada pipeline no treino completo e avaliar no teste isolado
print("=== AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (20%) ===\n")

resultados_teste = {}
for nome, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados_teste[nome] = {
        'Acurácia'       : accuracy_score(y_test, y_pred),
        'Precisão Macro' : precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall Macro'   : recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-Score Macro' : f1_score(y_test, y_pred, average='macro', zero_division=0),
    }

df_teste = pd.DataFrame(resultados_teste).T.round(4)
print(df_teste.to_string())

melhor_modelo_nome = df_teste['F1-Score Macro'].idxmax()
print(f"\n>> Melhor modelo: {melhor_modelo_nome} (F1-Score Macro = {df_teste.loc[melhor_modelo_nome, 'F1-Score Macro']:.4f})")


In [ ]:
# Matrizes de confusão para os 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
categorias = sorted(y.unique())

for idx, (nome, pipe) in enumerate(pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=categorias)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=[c[:8] for c in categorias],
                yticklabels=[c[:8] for c in categorias],
                ax=axes[idx], vmin=0, vmax=1)
    axes[idx].set_title(f'{nome}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predito')
    axes[idx].set_ylabel('Real')

plt.suptitle('Matrizes de Confusão Normalizadas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matrizes_confusao.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nAnálise das confusões:")
print("  - POLITICS x ENTERTAINMENT: manchetes sobre celebridades políticas")
print("  - WELLNESS x STYLE & BEAUTY: vocabulário parcialmente sobreposto (corpo, saúde, beleza)")
print("  - TRAVEL: menos confusão — vocabulário muito específico (hotéis, destinos, cidades)")


In [ ]:
# Classification report detalhado do melhor modelo
melhor_pipe = pipelines[melhor_modelo_nome]
y_pred_melhor = melhor_pipe.predict(X_test)

print(f"=== CLASSIFICATION REPORT — {melhor_modelo_nome} ===\n")
print(classification_report(y_test, y_pred_melhor, target_names=categorias, zero_division=0))


## 8. Feature Importance — Coeficientes do LinearSVC

> **NOTA:** Esta análise usa os coeficientes do modelo treinado com TF-IDF (após o pipeline completo). É diferente da análise de frequência bruta feita na EDA (seção 3).

In [ ]:
# Extração dos coeficientes do LinearSVC (feature importance)
# NOTA: Esta lista reflete as palavras mais discriminativas segundo o modelo TF-IDF treinado.
# É diferente da EDA (seção 3), que usava frequência bruta com stop words manuais.

pipe_svc = pipelines['LinearSVC']
pipe_svc.fit(X_train, y_train)

tfidf_vocab = pipe_svc.named_steps['tfidf'].get_feature_names_out()
coefs = pipe_svc.named_steps['clf'].coef_
classes = pipe_svc.named_steps['clf'].classes_

n_top = 10
fig, axes = plt.subplots(1, len(classes), figsize=(20, 5))

for idx, (cat, coef) in enumerate(zip(classes, coefs)):
    top_idx = np.argsort(coef)[-n_top:][::-1]
    top_words = tfidf_vocab[top_idx]
    top_vals  = coef[top_idx]

    axes[idx].barh(top_words[::-1], top_vals[::-1], color='#2E75B6')
    axes[idx].set_title(cat.replace(' & ', '\n& '), fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Coeficiente SVC')

plt.suptitle('Feature Importance — Top 10 Termos por Categoria (coeficientes LinearSVC + TF-IDF)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Impacto do class_weight='balanced'

In [ ]:
# Comparação com e sem class_weight para LinearSVC
print("=== Impacto do class_weight='balanced' no LinearSVC ===\n")

pipe_sem = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=50000)),
    ('clf', LinearSVC(random_state=42, max_iter=2000))
])
pipe_com = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=50000)),
    ('clf', LinearSVC(random_state=42, max_iter=2000, class_weight='balanced'))
])

pipe_sem.fit(X_train, y_train)
pipe_com.fit(X_train, y_train)

pred_sem = pipe_sem.predict(X_test)
pred_com = pipe_com.predict(X_test)

f1_sem = f1_score(y_test, pred_sem, average=None, labels=categorias, zero_division=0)
f1_com = f1_score(y_test, pred_com, average=None, labels=categorias, zero_division=0)

df_balance = pd.DataFrame({
    'Sem balanced': f1_sem,
    'Com balanced': f1_com,
}, index=categorias)

print(df_balance.round(4).to_string())
print(f"\nF1-Score Macro sem balanced : {f1_score(y_test, pred_sem, average='macro', zero_division=0):.4f}")
print(f"F1-Score Macro com balanced : {f1_score(y_test, pred_com, average='macro', zero_division=0):.4f}")

# Gráfico comparativo
ax = df_balance.plot(kind='bar', figsize=(10, 5), color=['#C0392B', '#1F4E79'], alpha=0.85)
ax.set_title("F1-Score por Categoria — Impacto do class_weight='balanced'", fontsize=12, fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('F1-Score')
ax.set_xticklabels(categorias, rotation=20, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('class_weight_comparacao.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Salvamento do Modelo Final

In [ ]:
import os

# Criar pasta model/ se não existir
os.makedirs('model', exist_ok=True)

# Salvar o Pipeline COMPLETO (TF-IDF + classificador juntos).
# É fundamental salvar o pipeline inteiro: o app Streamlit recebe texto cru
# e o pipeline aplica o TF-IDF automaticamente antes de classificar.
melhor_pipe.fit(X_train, y_train)  # treino no conjunto completo de treino
joblib.dump(melhor_pipe, 'model/modelo_final.joblib')
print(f"Modelo '{melhor_modelo_nome}' salvo em model/modelo_final.joblib")

# --- Verificação de carregamento ---
modelo_carregado = joblib.load('model/modelo_final.joblib')
exemplos = [
    "Trump signs new immigration bill in Senate",
    "Best yoga poses for stress relief and mental health",
    "Top 10 beaches to visit in Europe this summer",
    "New Marvel movie breaks box office record",
    "How to choose the best skin care routine"
]
predicoes = modelo_carregado.predict(exemplos)
print("\nTeste de carregamento — predições:")
for manchete, cat in zip(exemplos, predicoes):
    print(f"  [{cat}] {manchete}")
